# Musicm8 — free GPU training

Train the tiny Musicm8 model on audio you are authorized to use. The first milestone is deliberately small: tokenize a few tracks, overfit them, save `latest.pt`, and generate a short WAV.

> Free Colab GPU availability and session limits can change.

### Before running
1. Choose **Runtime → Change runtime type → GPU**.
2. Put audio in `MyDrive/Musicm8/audio/`.
3. Run the cells in order.


In [ ]:
import os, subprocess, sys, torch
print('Python:', sys.version)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. Choose Runtime > Change runtime type > GPU, then reconnect.')
print('GPU:', torch.cuda.get_device_name(0))


## 1. Mount Drive and clone/update Musicm8


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# Important: move out of /content/Musicm8 before deleting it, so this cell is safe to rerun.
%cd /content
!rm -rf /content/Musicm8
!git clone --depth 1 https://github.com/Elephant-logic/Musicm8.git /content/Musicm8
%cd /content/Musicm8
!python -m pip install -q --upgrade pip
!pip install -q -r requirements.txt
print('Musicm8 ready:', os.getcwd())


## 2. Configure the run
Start with these defaults.


In [ ]:
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/Musicm8')
AUDIO_DIR = DRIVE_ROOT / 'audio'
WORK_DIR = DRIVE_ROOT / 'work'
MANIFEST = DRIVE_ROOT / 'manifest.jsonl'
CODEC = 'encodec24'
CLIP_SECONDS = 8
STRIDE_SECONDS = 8
STEPS = 2000
BATCH_SIZE = 2
GRAD_ACCUM = 2
SAVE_EVERY = 250
RUN_NAME = 'tiny-overfit'
WORK_DIR.mkdir(parents=True, exist_ok=True)
AUDIO_DIR.mkdir(parents=True, exist_ok=True)
print('Audio folder:', AUDIO_DIR)
print('Work folder :', WORK_DIR)


## 3. Build/use the manifest


In [ ]:
import json
exts = {'.wav','.mp3','.flac','.m4a','.ogg','.aac'}
files = sorted(p for p in AUDIO_DIR.rglob('*') if p.suffix.lower() in exts)
print('Audio files found:', len(files))
if not files:
    raise FileNotFoundError(f'No audio found in {AUDIO_DIR}')
if not MANIFEST.exists():
    with MANIFEST.open('w', encoding='utf-8') as f:
        for p in files:
            caption = p.stem.replace('_',' ').replace('-',' ')
            f.write(json.dumps({'audio': str(p), 'caption': f'music track, {caption}'}, ensure_ascii=False) + '\n')
    print('Created manifest:', MANIFEST)
else:
    print('Using existing manifest:', MANIFEST)
print(MANIFEST.read_text(encoding='utf-8')[:1500])


## 4. Tokenize audio
The cache is stored in Drive so later sessions can reuse it.


In [ ]:
TOKENS_DIR = WORK_DIR / f'tokens-{CODEC}'
INDEX = TOKENS_DIR / 'index.jsonl'
if INDEX.exists() and INDEX.stat().st_size > 0:
    print('Token index already exists:', INDEX)
else:
    cmd = [sys.executable, 'tokenize_dataset.py', '--manifest', str(MANIFEST), '--out', str(TOKENS_DIR), '--codec', CODEC, '--channels', '1', '--clip-seconds', str(CLIP_SECONDS), '--stride-seconds', str(STRIDE_SECONDS), '--keep-tail', '--device', 'cuda']
    print('Running:', ' '.join(cmd))
    result = subprocess.run(cmd)
    if result.returncode != 0:
        raise RuntimeError(f'Tokenization failed with exit code {result.returncode}. Scroll up to the FIRST traceback/error above this line.')
print('Tokenization ready:', INDEX)


## 5. Train / resume


In [ ]:
RUN_DIR = WORK_DIR / 'runs' / RUN_NAME
LATEST = RUN_DIR / 'latest.pt'
cmd = [sys.executable, 'train.py', '--data', str(INDEX), '--config', 'configs/v2-tiny.json', '--out', str(RUN_DIR), '--batch-size', str(BATCH_SIZE), '--grad-accum', str(GRAD_ACCUM), '--steps', str(STEPS), '--save-every', str(SAVE_EVERY), '--num-workers', '2', '--device', 'cuda']
if LATEST.exists():
    cmd += ['--resume', str(LATEST)]
    print('Resuming:', LATEST)
else:
    print('Starting a fresh run')
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)
print('Checkpoint:', LATEST)


## 6. Generate a sample


In [ ]:
SAMPLE = WORK_DIR / 'sample.wav'
PROMPT = 'dark atmospheric electronic music with deep bass and wide synth pads'
cmd = [sys.executable, 'generate.py', '--checkpoint', str(LATEST), '--prompt', PROMPT, '--seconds', '8', '--seed', '42', '--out', str(SAMPLE)]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)
from IPython.display import Audio, display
display(Audio(str(SAMPLE)))
print('Saved:', SAMPLE)
